In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.model_selection import train_test_split
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from chemprop import data, featurizers, models, nn
from rdkit.Chem import MolFromSmiles
import concurrent.futures
import logging
from chemprop.data import datapoints, dataloader, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer
from chemprop.data.datapoints import MoleculeDatapoint
import torch
import chemprop.nn.metrics as chem_metrics
from chemprop.nn.agg import (
    MultiHeadAttentiveAggregation,
    GatedAttentiveAggregation,
    GatedAttentiveAggregationv2
)
from assets.functionchem import *
from assets.chempropcombination import *

df = pd.read_csv("./data/pkaAcidicData.csv")

# Apply the scaffold computation for each SMILES
df['scaffold'] = df['smiles'].apply(compute_scaffold)

# Step 1: Get the unique scaffolds
scaffold_list = df['scaffold'].unique()

# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

train_df = train_df.rename(columns={"SMILES": "smiles", "pKa_Acidic": "pKa"})
val_df = val_df.rename(columns={"SMILES": "smiles", "pKa_Acidic": "pKa"})
test_df = test_df.rename(columns={"SMILES": "smiles", "pKa_Acidic": "pKa"})

results_df = run_chemprop_mp_agg_benchmark(
    train_df,
    val_df,
    test_df,
    target_column="pKa",
    max_epochs=200
)

[07:50:32] WARNING: not removing hydrogen atom without neighbors
Seed set to 42


Training set size: 4865
Test set size: 608
Validation set size: 609


[07:50:33] WARNING: not removing hydrogen atom without neighbors
Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 4865, batch_size = 64)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.



Training: BondMP3 + Mean


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

...

Output()

`Trainer.fit` stopped: `max_epochs=200` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Test: {'Target': 'pKa', 'MessagePassing': 'AtomMP6', 'Aggregation': 'MultiHead12', 'Test_MAE': 0.4474445581436157, 'Test_RMSE': 0.8844875693321228, 'Test_R2': 0.9596803784370422}

Final comparison:
   Target MessagePassing     Aggregation  Test_MAE  Test_RMSE   Test_R2
45    pKa        AtomMP6      MultiHead4  0.441084   0.890359  0.959143
47    pKa        AtomMP6     MultiHead12  0.447445   0.884488  0.959680
46    pKa        AtomMP6      MultiHead8  0.460213   0.903050  0.957970
17    pKa        BondMP5     MultiHead12  0.462609   1.038542  0.944412
9     pKa        BondMP4      MultiHead4  0.464282   0.965552  0.951951
31    pKa        AtomMP4  GatedAttentive  0.466806   0.915529  0.956801
40    pKa        AtomMP5      MultiHead8  0.468086   0.939153  0.954543
23    pKa        BondMP6     MultiHead12  0.471046   1.090489  0.938712
41    pKa        AtomMP5     MultiHead12  0.472453   0.948220  0.953661
15    pKa        BondMP5      MultiHead4  0.474826   1.000736  0.948385
16    pKa 